In [2]:
import pandas as pd
import win32com.client as win32
from datetime import date

# === Configuración ===
archivo = r"C:\Users\dany0\OneDrive - JEP Colombia\Archivos de Erica Velasquez Rodríguez - DESPLAZAMIENTOS Y COMISIONES 2026\2026 MATRIZ DE SEGUIMIENTO A DESPLAZAMIENTOS Y COMISIONES.xlsx"
hoja_base = "DESPLAZAMIENTOS 2026"
hoja_correos = "NOMBRES"
correo_notificacion = "daniel.perezd@jep.gov.co"

# === Leer datos ===
df_base = pd.read_excel(archivo, sheet_name="DESPLAZAMIENTOS_2026", header=0)
df_correos = pd.read_excel(archivo, sheet_name=hoja_correos, header=0)

# === Fecha actual ===
hoy = date.today()

# === Crear campos por posición ===
df_base["fecha_legalizacion"] = pd.to_datetime(df_base.iloc[:, 13], errors="coerce").dt.date
df_base["estado"] = df_base.iloc[:, 11].astype(str).str.strip().str.lower()
df_base["fin_semana"] = df_base.iloc[:, 29].astype(str).str.strip().str.upper()

# ID está en la columna 3 (índice 2)
df_base["ID_temp"] = df_base.iloc[:, 2]

# === FILTRO PRINCIPAL (NO incluye fin de semana) ===
registros_hoy = df_base[
    (df_base["fecha_legalizacion"] == hoy) &
    (df_base["estado"] == "radicada")
]

# === Conectar Outlook ===
outlook = win32.Dispatch("Outlook.Application")

if not registros_hoy.empty:

    enviados = []

    for _, fila in registros_hoy.iterrows():
        nombre = fila.iloc[0]
        id_registro = fila["ID_temp"]
        fin_semana = fila["fin_semana"]

        # buscar correo
        correo = df_correos.loc[df_correos["NOMBRE"] == nombre, "CORREOS"]

        if correo.empty:
            continue

        correo_funcionario = correo.values[0]

        # escoger mensaje según SI / NO
        if fin_semana == "SI":
            mensaje = (
                f"Cordial saludo {nombre},\n\n"
                f"Recuerde que debe enviar su informe de legalización a la Asesora III de la OAGT con copia a Erica Velásquez, correspondiente al ID {id_registro}.\n\n"
                "Así mismo recuerde que tiene derecho a disfrutar de permiso remunerado por los días laborados en fin de semana, "
                "este se debe solicitar y disfrutar dentro de los 30 días siguientes a la finalización de la comisión; "
                "para ello debe diligenciar el formato de solicitud de permiso y adjuntar el informe de legalización ya tramitado.\n\n"
                "Saludos."
            )
        else:
            mensaje = (
                f"Cordial saludo {nombre},\n\n"
                f"Recuerde que debe enviar su informe de legalización a la Asesora III de la OAGT con copia a Erica Velásquez, correspondiente al ID {id_registro}.\n\n"
                "Saludos."
            )

        # enviar correo
        mail = outlook.CreateItem(0)
        mail.To = correo_funcionario
        mail.CC = "erica.velasquez@jep.gov.co;claudia.aguirre@jep.gov.co;julieth.ramirez@jep.gov.co"
        mail.Subject = f"PLAZO MÁXIMO PARA RADICAR LEGALIZACIÓN A GLORIA CALA - ID {id_registro}"
        mail.Body = mensaje
        mail.Send()

        enviados.append(f"{nombre} (ID: {id_registro}, Fin de semana: {fin_semana})")

    # Notificación para ti
    mail_admin = outlook.CreateItem(0)
    mail_admin.To = correo_notificacion
    mail_admin.Subject = "✔ Reporte de envíos automáticos"
    mail_admin.Body = (
        f"Hoy ({hoy}) se enviaron {len(enviados)} correos.\n\n" +
        "\n".join(enviados)
    )
    mail_admin.Send()

    print("✔ Correos enviados y notificación enviada.")

else:
    # Notificación si no hubo registros
    mail_admin = outlook.CreateItem(0)
    mail_admin.To = correo_notificacion
    mail_admin.Subject = "📭 Reporte de envíos automáticos"
    mail_admin.Body = f"Hoy ({hoy}) no había registros con fecha de hoy."
    mail_admin.Send()

    print("📭 No hay registros para hoy.")


📭 No hay registros para hoy.


c:\Users\dany0\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [ ]:
import sys
!{sys.executable} -m pip install Unidecode # forma de instalar una libreria desde el jupyter

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
